In [ ]:
# =====================================================
# 1. IMPORTACIONES
# =====================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

print("Librerías importadas correctamente.")

In [ ]:
# =====================================================
# 2. CONFIGURACIÓN DE REPRODUCIBILIDAD
# =====================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("Semilla configurada:", SEED)

# Comprobar GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# =====================================================
# 3. LONGITUD MÁXIMA DE LAS SECUENCIAS
# =====================================================

MAX_LENGTH = 128

print("Longitud máxima:", MAX_LENGTH)

In [ ]:
# =====================================================
# 4. RUTAS GENERALES
# =====================================================

# Datos
RUTA_DATOS = r"./Bases de Datos Splits/"

# Resultados de Optuna
RUTA_OPTUNA = r"./Resultados Optuna/"

# Checkpoints temporales del entrenamiento
RUTA_CHECKPOINTS = r"./Checkpoints/"

# Modelos finales entrenados
RUTA_MODELOS = r"./Modelos Definitivos/"

# Tablas, métricas y figuras finales
RUTA_RESULTADOS = r"./Resultados Definitivos/"


# Crear automáticamente las carpetas de salida
os.makedirs(RUTA_CHECKPOINTS, exist_ok=True)
os.makedirs(RUTA_MODELOS, exist_ok=True)
os.makedirs(RUTA_RESULTADOS, exist_ok=True)


# -------------------------
# DATD
# -------------------------

ruta_train_DATD = RUTA_DATOS + "train_DATD_balanceado.csv"
ruta_val_DATD   = RUTA_DATOS + "val_DATD.csv"
ruta_test_DATD  = RUTA_DATOS + "test_DATD.csv"

# -------------------------
# SDCNL
# -------------------------

ruta_train_SDCNL = RUTA_DATOS + "train_SDCNL.csv"
ruta_val_SDCNL   = RUTA_DATOS + "val_SDCNL.csv"
ruta_test_SDCNL  = RUTA_DATOS + "test_SDCNL.csv"

# -------------------------
# DU
# -------------------------

ruta_train_DU = RUTA_DATOS + "train_DU_balanceado.csv"
ruta_val_DU   = RUTA_DATOS + "val_DU.csv"
ruta_test_DU  = RUTA_DATOS + "test_DU.csv"

print("Rutas configuradas correctamente.")

In [ ]:
# =====================================================
# 5. CARGA DE LOS DATASETS
# =====================================================

# DATD
train_DATD = pd.read_csv(ruta_train_DATD)
val_DATD   = pd.read_csv(ruta_val_DATD)
test_DATD  = pd.read_csv(ruta_test_DATD)

# SDCNL
train_SDCNL = pd.read_csv(ruta_train_SDCNL)
val_SDCNL   = pd.read_csv(ruta_val_SDCNL)
test_SDCNL  = pd.read_csv(ruta_test_SDCNL)

# DU
train_DU = pd.read_csv(ruta_train_DU)
val_DU   = pd.read_csv(ruta_val_DU)
test_DU  = pd.read_csv(ruta_test_DU)

print("Datasets cargados correctamente.")

In [ ]:
# =====================================================
# 6. COMPROBACIÓN DE LOS DATASETS
# =====================================================

datasets = {
    "DATD Train": train_DATD,
    "DATD Validation": val_DATD,
    "DATD Test": test_DATD,

    "SDCNL Train": train_SDCNL,
    "SDCNL Validation": val_SDCNL,
    "SDCNL Test": test_SDCNL,

    "DU Train": train_DU,
    "DU Validation": val_DU,
    "DU Test": test_DU
}

for nombre, df in datasets.items():

    print("\n========================================")
    print(nombre)
    print("========================================")

    print("Instancias:", len(df))
    print("Columnas:", df.columns.tolist())

    if "Label" in df.columns:
        print("\nDistribución de clases:")
        print(df["Label"].value_counts().sort_index())

# BLOQUE 2. Elección de Transformer y covertir los Datos

In [ ]:
# =====================================================
# 7. SELECCIÓN DEL TRANSFORMER
# =====================================================

# Modelo 1
# nombre_modelo = "roberta-base"

# Modelo 2
nombre_modelo = "microsoft/deberta-v3-base"

if nombre_modelo == "roberta-base":
    nombre_corto = "RoBERTa"
elif nombre_modelo == "microsoft/deberta-v3-base":
    nombre_corto = "DeBERTa"
else:
    raise ValueError("Transformer no reconocido.")

print("Transformer seleccionado:", nombre_modelo)
print("Nombre corto:", nombre_corto)

In [ ]:
# =====================================================
# 8. CARGA DEL TOKENIZER
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

print("Tokenizer cargado correctamente.")
print("Transformer:", nombre_modelo)
print("Longitud máxima utilizada:", MAX_LENGTH)

In [ ]:
# =====================================================
# 9. FUNCIÓN DE TOKENIZACIÓN
# =====================================================

def tokenizar_textos(df):

    return tokenizer(
        df["Text"].astype(str).tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

print("Función de tokenización creada correctamente.")

In [ ]:
# =====================================================
# 10. DATASET PARA HUGGING FACE
# =====================================================

class DatasetTexto(torch.utils.data.Dataset):

    def __init__(self, codificaciones, etiquetas):
        self.codificaciones = codificaciones
        self.etiquetas = etiquetas

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, idx):

        item = {
            key: value[idx]
            for key, value in self.codificaciones.items()
        }

        item["labels"] = torch.tensor(
            self.etiquetas[idx],
            dtype=torch.long
        )

        return item

print("Clase DatasetTexto creada correctamente.")

In [ ]:
# =====================================================
# 11. TOKENIZACIÓN DE LOS DATASETS
# =====================================================

print("Tokenizando DATD...")
tokens_train_DATD = tokenizar_textos(train_DATD)
tokens_val_DATD   = tokenizar_textos(val_DATD)
tokens_test_DATD  = tokenizar_textos(test_DATD)

print("Tokenizando SDCNL...")
tokens_train_SDCNL = tokenizar_textos(train_SDCNL)
tokens_val_SDCNL   = tokenizar_textos(val_SDCNL)
tokens_test_SDCNL  = tokenizar_textos(test_SDCNL)

print("Tokenizando DU...")
tokens_train_DU = tokenizar_textos(train_DU)
tokens_val_DU   = tokenizar_textos(val_DU)
tokens_test_DU  = tokenizar_textos(test_DU)

print("\nTodos los datasets han sido tokenizados correctamente.")

In [ ]:
# =====================================================
# 12. PREPARACIÓN DE LAS ETIQUETAS
# =====================================================

# -------------------------
# DATD
# -------------------------

labels_train_DATD = train_DATD["Label"].astype(int).tolist()
labels_val_DATD   = val_DATD["Label"].astype(int).tolist()
labels_test_DATD  = test_DATD["Label"].astype(int).tolist()


# -------------------------
# SDCNL
# -------------------------
# Etiquetas originales:
# 1 = depresión
# 2 = suicidio
#
# Para el clasificador binario:
# 1 -> 0
# 2 -> 1

labels_train_SDCNL = (train_SDCNL["Label"].astype(int) - 1).tolist()
labels_val_SDCNL   = (val_SDCNL["Label"].astype(int) - 1).tolist()
labels_test_SDCNL  = (test_SDCNL["Label"].astype(int) - 1).tolist()


# -------------------------
# DU
# -------------------------

labels_train_DU = train_DU["Label"].astype(int).tolist()
labels_val_DU   = val_DU["Label"].astype(int).tolist()
labels_test_DU  = test_DU["Label"].astype(int).tolist()

print("Etiquetas preparadas correctamente.")

In [ ]:
# =====================================================
# 13. CREACIÓN DE LOS DATASETS PARA TRAINER
# =====================================================

# DATD
dataset_train_DATD = DatasetTexto(
    tokens_train_DATD,
    labels_train_DATD
)

dataset_val_DATD = DatasetTexto(
    tokens_val_DATD,
    labels_val_DATD
)

dataset_test_DATD = DatasetTexto(
    tokens_test_DATD,
    labels_test_DATD
)


# SDCNL
dataset_train_SDCNL = DatasetTexto(
    tokens_train_SDCNL,
    labels_train_SDCNL
)

dataset_val_SDCNL = DatasetTexto(
    tokens_val_SDCNL,
    labels_val_SDCNL
)

dataset_test_SDCNL = DatasetTexto(
    tokens_test_SDCNL,
    labels_test_SDCNL
)


# DU
dataset_train_DU = DatasetTexto(
    tokens_train_DU,
    labels_train_DU
)

dataset_val_DU = DatasetTexto(
    tokens_val_DU,
    labels_val_DU
)

dataset_test_DU = DatasetTexto(
    tokens_test_DU,
    labels_test_DU
)

print("Datasets para Trainer creados correctamente.")

In [ ]:
# =====================================================
# 14. COMPROBACIÓN FINAL
# =====================================================

print("DATD")
print("Train:", len(dataset_train_DATD))
print("Validation:", len(dataset_val_DATD))
print("Test:", len(dataset_test_DATD))

print("\nSDCNL")
print("Train:", len(dataset_train_SDCNL))
print("Validation:", len(dataset_val_SDCNL))
print("Test:", len(dataset_test_SDCNL))

print("\nDU")
print("Train:", len(dataset_train_DU))
print("Validation:", len(dataset_val_DU))
print("Test:", len(dataset_test_DU))

print("\nEtiquetas SDCNL utilizadas por el modelo:")
print("Train:", sorted(set(labels_train_SDCNL)))
print("Validation:", sorted(set(labels_val_SDCNL)))
print("Test:", sorted(set(labels_test_SDCNL)))

# BLOQUE 3. Uso de Hiperparámetros encontrados por Optuna

In [ ]:
# =====================================================
# 15. FUNCIÓN PARA CARGAR LOS MEJORES HIPERPARÁMETROS
# =====================================================

def cargar_mejores_hiperparametros(nombre_modelo, dataset):

    # Crear un nombre corto para identificar el Transformer
    if nombre_modelo == "roberta-base":
        nombre_corto = "RoBERTa"

    elif nombre_modelo == "microsoft/deberta-v3-base":
        nombre_corto = "DeBERTa"

    else:
        raise ValueError("Transformer no reconocido.")

    # Construir la ruta del CSV correspondiente
    ruta_csv = RUTA_OPTUNA + f"resultados_optuna_{nombre_corto}_{dataset}.csv"

    # Cargar los resultados de Optuna
    resultados = pd.read_csv(ruta_csv)

    # Seleccionar el trial que obtuvo el mayor valor objetivo
    mejor_trial = resultados.loc[resultados["value"].idxmax()]

    # Extraer los hiperparámetros ganadores
    mejores_hp = {
        "batch_size": int(mejor_trial["params_batch_size"]),
        "learning_rate": float(mejor_trial["params_learning_rate"]),
        "weight_decay": float(mejor_trial["params_weight_decay"]),
        "epochs": int(mejor_trial["params_epochs"])
    }

    return mejores_hp

In [ ]:
# =====================================================
# 16. CARGAR HIPERPARÁMETROS GANADORES
# =====================================================

hp_DATD = cargar_mejores_hiperparametros(
    nombre_modelo,
    "DATD"
)

hp_SDCNL = cargar_mejores_hiperparametros(
    nombre_modelo,
    "SDCNL"
)

hp_DU = cargar_mejores_hiperparametros(
    nombre_modelo,
    "DU"
)

print("========== DATD ==========")
print(hp_DATD)

print("\n========== SDCNL ==========")
print(hp_SDCNL)

print("\n========== DU ==========")
print(hp_DU)

# BLOQUE 4. Definir la métrica que se usará durante la evaluación del entrenamiento

In [ ]:
# =====================================================
# 17. MÉTRICAS PARA EL FINE-TUNING
# =====================================================

# Para las tareas Binarias - DATD y SDCNL
def compute_metrics_binario(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1 = f1_score(
        labels,
        predictions,
        average="binary",
        pos_label=1
    )

    return {"f1": f1}

# Para las tareas Multiclase - DU
def compute_metrics_multiclase(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {"f1": f1}

print("Funciones de métricas preparadas correctamente.")

# BLOQUE 5. USO DEL FINE-TUNING Y ENTRENO DE MODELOS

In [ ]:
# =============================================================================
# 18. FUNCIÓN GENERAL PARA ENTRENAR Y EVALUAR CADA MODELO FINAL POR SEPARADO
# =============================================================================

# Esta función se utilizará tres veces para cada Transformer:
#
# 1) DATD  -> entrenar el modelo de NIVEL 1:
#             sano vs enfermo
#
# 2) SDCNL -> entrenar el modelo de NIVEL 2:
#             depresión vs suicidio
#
# 3) DU    -> entrenar el MODELO DIRECTO:
#             sano vs depresión vs suicidio
#
# IMPORTANTE:
# Esta función NO crea la cascada.
# Aquí los tres modelos se entrenan y evalúan por separado.
# La cascada se construirá más adelante utilizando
# los modelos DATD y SDCNL ya entrenados.

def entrenar_modelo_definitivo(
    nombre_tarea,
    num_labels,
    dataset_train,
    dataset_val,
    dataset_test,
    compute_metrics,
    hiperparametros
):

    # Reiniciar la semilla antes de crear cada modelo
    set_seed(SEED)

    print("\n========================================")
    print("ENTRENAMIENTO DEL MODELO:", nombre_tarea)
    print("Transformer:", nombre_modelo)
    print("========================================")

    # Explicar qué modelo se está entrenando
    if nombre_tarea == "DATD":
        print(
            "Función del modelo: NIVEL 1 de la cascada "
            "(sano vs enfermo)"
        )

    elif nombre_tarea == "SDCNL":
        print(
            "Función del modelo: NIVEL 2 de la cascada "
            "(depresión vs suicidio)"
        )

    elif nombre_tarea == "DU":
        print(
            "Función del modelo: MODELO DIRECTO "
            "(sano vs depresión vs suicidio)"
        )

    print("\nHiperparámetros utilizados:")
    print(hiperparametros)

    # =================================================
    # CREAR EL TRANSFORMER
    # =================================================

    # Se parte otra vez del Transformer preentrenado base
    # y se crea una nueva cabeza de clasificación
    model = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=num_labels,
        dtype=torch.float32
    )

    # =================================================
    # CONFIGURACIÓN DEL ENTRENAMIENTO
    # =================================================

    args = TrainingArguments(

        # Carpeta temporal para checkpoints
        output_dir=os.path.join(
            RUTA_CHECKPOINTS,
            nombre_corto,
            nombre_tarea
        ),

        # Evaluar y guardar al final de cada época
        eval_strategy="epoch",
        save_strategy="epoch",

        # Validation se utiliza para seleccionar
        # la mejor época del entrenamiento
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        # Hiperparámetros ganadores obtenidos con Optuna
        learning_rate=hiperparametros["learning_rate"],
        per_device_train_batch_size=hiperparametros["batch_size"],
        per_device_eval_batch_size=16,
        weight_decay=hiperparametros["weight_decay"],
        num_train_epochs=hiperparametros["epochs"],

        # Configuración común
        warmup_steps=100,
        max_grad_norm=1.0,

        # Reproducibilidad
        seed=SEED,

        # Conservar únicamente el mejor checkpoint
        save_total_limit=1,

        logging_strategy="epoch",
        report_to="none"
    )

    # =================================================
    # CREAR TRAINER
    # =================================================

    trainer = Trainer(
        model=model,
        args=args,

        # TRAIN:
        # aquí están los ejemplos que el modelo utiliza
        # para actualizar sus pesos y aprender
        train_dataset=dataset_train,

        # VALIDATION:
        # no se usa para actualizar pesos;
        # sirve para seleccionar la mejor época
        eval_dataset=dataset_val,

        compute_metrics=compute_metrics,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=3
            )
        ]
    )

    # =================================================
    # ENTRENAMIENTO REAL
    # =================================================

    print("\nComenzando entrenamiento con TRAIN...")

    # AQUÍ ES DONDE EL MODELO APRENDE
    trainer.train()

    # Al terminar, Trainer recupera automáticamente
    # el checkpoint que obtuvo el mejor F1 en Validation
    mejor_f1_val = trainer.state.best_metric

    print("\nEntrenamiento finalizado.")
    print("Mejor F1 en Validation:", mejor_f1_val)

    # =================================================
    # EVALUACIÓN FINAL SOBRE TEST
    # =================================================

    # IMPORTANTE:
    # aquí el modelo YA NO aprende.
    # Test solo se utiliza para medir el rendimiento
    # del modelo ya entrenado y seleccionado.
    resultados_test = trainer.evaluate(dataset_test)

    f1_test = resultados_test["eval_f1"]

    print("F1 final en Test:", f1_test)

    # =================================================
    # GUARDAR EL MODELO ENTRENADO
    # =================================================

    ruta_modelo = os.path.join(
        RUTA_MODELOS,
        f"{nombre_corto}_{nombre_tarea}"
    )

    trainer.save_model(ruta_modelo)
    tokenizer.save_pretrained(ruta_modelo)

    print("Modelo entrenado guardado en:")
    print(ruta_modelo)

    # =================================================
    # LIBERAR MEMORIA
    # =================================================

    del trainer
    del model

    gc.collect()
    torch.cuda.empty_cache()

    return mejor_f1_val, f1_test, ruta_modelo

In [ ]:
# =================================================================================
# 19. ENTRENAMIENTO DE LOS TRES MODELOS FINALES DEL TRANSFORMER SELECCIONADO
# =================================================================================

# =====================================================
# MODELO 1
# DATD -> NIVEL 1 DE LA CASCADA
# =====================================================

print("\n\n########################################")
print("MODELO 1 DE 3")
print("DATD -> NIVEL 1 DE LA CASCADA")
print("Tarea: SANO vs ENFERMO")
print("########################################")

f1_val_DATD, f1_test_DATD, ruta_DATD = entrenar_modelo_definitivo(
    "DATD",
    2,
    dataset_train_DATD,
    dataset_val_DATD,
    dataset_test_DATD,
    compute_metrics_binario,
    hp_DATD
)


# =====================================================
# MODELO 2
# SDCNL -> NIVEL 2 DE LA CASCADA
# =====================================================

print("\n\n########################################")
print("MODELO 2 DE 3")
print("SDCNL -> NIVEL 2 DE LA CASCADA")
print("Tarea: DEPRESIÓN vs SUICIDIO")
print("########################################")

f1_val_SDCNL, f1_test_SDCNL, ruta_SDCNL = entrenar_modelo_definitivo(
    "SDCNL",
    2,
    dataset_train_SDCNL,
    dataset_val_SDCNL,
    dataset_test_SDCNL,
    compute_metrics_binario,
    hp_SDCNL
)


# =====================================================
# MODELO 3
# DU -> MODELO DIRECTO
# =====================================================

print("\n\n########################################")
print("MODELO 3 DE 3")
print("DU -> MODELO DIRECTO")
print("Tarea: SANO vs DEPRESIÓN vs SUICIDIO")
print("########################################")

f1_val_DU, f1_test_DU, ruta_DU = entrenar_modelo_definitivo(
    "DU",
    3,
    dataset_train_DU,
    dataset_val_DU,
    dataset_test_DU,
    compute_metrics_multiclase,
    hp_DU
)


print("\n========================================")
print("ENTRENAMIENTO COMPLETADO")
print("========================================")

print(
    f"{nombre_corto}_DATD  -> modelo de Nivel 1 entrenado"
)

print(
    f"{nombre_corto}_SDCNL -> modelo de Nivel 2 entrenado"
)

print(
    f"{nombre_corto}_DU    -> modelo DIRECTO entrenado"
)

print("\nIMPORTANTE:")
print(
    "DATD y SDCNL se utilizarán más adelante para construir "
    "las distintas cascadas."
)

print(
    "El resultado de DU sobre test_DU ya corresponde "
    "al resultado final del modelo directo."
)

In [ ]:
# ================================================================
# 20. RESULTADOS DE LOS TRES MODELOS ENTRENADOS POR SEPARADO
# ================================================================

resultados_modelos_individuales = pd.DataFrame({

    "Transformer": [
        nombre_corto,
        nombre_corto,
        nombre_corto
    ],

    "Modelo_entrenado": [
        f"{nombre_corto}_DATD",
        f"{nombre_corto}_SDCNL",
        f"{nombre_corto}_DU"
    ],

    "Funcion": [
        "Nivel 1 de la cascada",
        "Nivel 2 de la cascada",
        "Modelo directo"
    ],

    "Tarea": [
        "Sano vs Enfermo",
        "Depresión vs Suicidio",
        "Sano vs Depresión vs Suicidio"
    ],

    "Dataset_Test": [
        "test_DATD",
        "test_SDCNL",
        "test_DU"
    ],

    "F1_Validation": [
        f1_val_DATD,
        f1_val_SDCNL,
        f1_val_DU
    ],

    "F1_Test": [
        f1_test_DATD,
        f1_test_SDCNL,
        f1_test_DU
    ]
})


print("\n========================================")
print("RESULTADOS DE LOS MODELOS ENTRENADOS")
print("========================================")

display(resultados_modelos_individuales)


print("\nInterpretación:")
print(
    "- DATD: modelo entrenado para actuar como Nivel 1 "
    "de la futura cascada."
)

print(
    "- SDCNL: modelo entrenado para actuar como Nivel 2 "
    "de la futura cascada."
)

print(
    "- DU: modelo directo completo. Su F1 sobre test_DU "
    "sí es ya un resultado final del enfoque directo."
)

print(
    "- Los F1 de DATD y SDCNL todavía NO son "
    "resultados de una cascada."
)

In [ ]:
# =====================================================
# 21. GUARDAR RESULTADOS INDIVIDUALES DEL TRANSFORMER
# =====================================================

# Guardar la tabla de resultados individuales
resultados_modelos_individuales.to_csv(
    RUTA_RESULTADOS + f"resultados_modelos_individuales_{nombre_corto}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Resultados individuales de {nombre_corto} guardados correctamente.")

In [ ]:
# =====================================================
# 22. PARADA ANTES DE LA EVALUACIÓN CONJUNTA
# =====================================================

raise RuntimeError(
    "PARADA INTENCIONADA: el Transformer seleccionado ya ha sido "
    "entrenado y evaluado. Antes de continuar deben haberse ejecutado "
    "tanto RoBERTa como DeBERTa. Cuando existan los 6 modelos finales, "
    "continúa manualmente desde la siguiente celda."
)

In [ ]:
# ============================================================
# 23. RESUMEN DE RESULTADOS INDIVIDUALES ROBERTA Y DEBERTA
# ============================================================

# Cargar los resultados individuales de ambos Transformers
resultados_roberta = pd.read_csv(
    RUTA_RESULTADOS + "resultados_modelos_individuales_RoBERTa.csv"
)

resultados_deberta = pd.read_csv(
    RUTA_RESULTADOS + "resultados_modelos_individuales_DeBERTa.csv"
)

# Unir ambos resultados en una única tabla
resumen_resultados_modelos_individuales = pd.concat(
    [
        resultados_roberta,
        resultados_deberta
    ],
    ignore_index=True
)

print("\n========================================")
print("RESUMEN DE RESULTADOS INDIVIDUALES")
print("========================================")

display(resumen_resultados_modelos_individuales)


resumen_resultados_modelos_individuales.to_csv(
    RUTA_RESULTADOS + "resumen_resultados_modelos_individuales.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resumen conjunto guardado correctamente.")

# BLOQUE 6. Construir el sistema en cascada

In [ ]:
# =====================================================
# 24. FUNCIÓN PARA CARGAR UN MODELO ENTRENADO
# =====================================================

# Esta funcion permite abrir desde disco uno de los seis modelos ya entrenados.

def cargar_modelo_entrenado(ruta_modelo):

    tokenizer_modelo = AutoTokenizer.from_pretrained(
        ruta_modelo
    )

    modelo = AutoModelForSequenceClassification.from_pretrained(
        ruta_modelo,
        dtype=torch.float32
    )

    modelo.to(device)
    modelo.eval()

    return modelo, tokenizer_modelo


print("Función de carga preparada correctamente.")

In [ ]:
# =====================================================
# 25. FUNCIÓN DE PREDICCIÓN
# =====================================================

def predecir_textos(
    modelo,
    tokenizer_modelo,
    textos,
    batch_size=16
):

    predicciones = []

    # Recorrer los textos por lotes
    for inicio in range(0, len(textos), batch_size):

        lote_textos = textos[inicio:inicio + batch_size]

        # Tokenizar el lote
        entradas = tokenizer_modelo(
            lote_textos,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        # Enviar los datos a GPU
        entradas = {
            clave: valor.to(device)
            for clave, valor in entradas.items()
        }

        # Desactivar cálculo de gradientes
        # porque estamos haciendo predicción, no entrenamiento
        with torch.no_grad():

            salidas = modelo(**entradas)

            pred_lote = torch.argmax(
                salidas.logits,
                dim=-1
            )

        predicciones.extend(
            pred_lote.cpu().numpy().tolist()
        )

    return predicciones


print("Función de predicción preparada correctamente.")

In [ ]:
# =====================================================
# 26. FUNCIÓN DE ENCADENAMIENTO DE CASCADA
# =====================================================

def predecir_cascada(
    textos,
    modelo_n1,
    tokenizer_n1,
    modelo_n2,
    tokenizer_n2
):

    # =================================================
    # NIVEL 1
    # =================================================

    # Predecir todos los textos con DATD
    pred_n1 = predecir_textos(
        modelo_n1,
        tokenizer_n1,
        textos
    )

    # Inicializar predicciones finales
    pred_finales = np.zeros(
        len(textos),
        dtype=int
    )

    # Buscar qué textos han sido clasificados
    # como "enfermos" por el Nivel 1
    indices_enfermos = [
        i
        for i, pred in enumerate(pred_n1)
        if pred == 1
    ]

    print("Total de textos:", len(textos))
    print("Predichos como sanos:", pred_n1.count(0))
    print("Enviados al Nivel 2:", len(indices_enfermos))

    # =================================================
    # NIVEL 2
    # =================================================

    if len(indices_enfermos) > 0:

        # Seleccionar únicamente los textos considerados enfermos
        textos_enfermos = [
            textos[i]
            for i in indices_enfermos
        ]

        # Clasificarlos con SDCNL
        pred_n2 = predecir_textos(
            modelo_n2,
            tokenizer_n2,
            textos_enfermos
        )

        # Recordatorio:
        #
        # salida interna SDCNL:
        # 0 = depresión
        # 1 = suicidio
        #
        # etiqueta final DU:
        # 1 = depresión
        # 2 = suicidio

        for indice_original, pred in zip(
            indices_enfermos,
            pred_n2
        ):

            pred_finales[indice_original] = pred + 1

    return pred_finales

In [ ]:
# =====================================================
# 27. RUTAS DE LOS MODELOS DEFINITIVOS
# =====================================================

rutas_modelos = {
    "RoBERTa_DATD": RUTA_MODELOS + "RoBERTa_DATD",
    "RoBERTa_SDCNL": RUTA_MODELOS + "RoBERTa_SDCNL",
    "RoBERTa_DU": RUTA_MODELOS + "RoBERTa_DU",

    "DeBERTa_DATD": RUTA_MODELOS + "DeBERTa_DATD",
    "DeBERTa_SDCNL": RUTA_MODELOS + "DeBERTa_SDCNL",
    "DeBERTa_DU": RUTA_MODELOS + "DeBERTa_DU"
}

print("Rutas de los modelos configuradas.")

In [ ]:
# =====================================================
# 28. FUNCIÓN PARA EVALUAR UNA CASCADA
# =====================================================

def evaluar_cascada(
    nombre_n1,
    ruta_n1,
    nombre_n2,
    ruta_n2,
    textos_test,
    etiquetas_reales
):

    print("\n========================================")
    print("EVALUANDO CASCADA")
    print("Nivel 1:", nombre_n1)
    print("Nivel 2:", nombre_n2)
    print("========================================")

    # Cargar el modelo del Nivel 1
    modelo_n1, tokenizer_n1 = cargar_modelo_entrenado(ruta_n1)

    # Cargar el modelo del Nivel 2
    modelo_n2, tokenizer_n2 = cargar_modelo_entrenado(ruta_n2)

    # Realizar la predicción completa en cascada
    predicciones = predecir_cascada(
        textos_test,
        modelo_n1,
        tokenizer_n1,
        modelo_n2,
        tokenizer_n2
    )

    # Calcular Macro-F1 porque el resultado final
    # tiene tres clases: 0, 1 y 2
    f1_cascada = f1_score(
        etiquetas_reales,
        predicciones,
        average="macro"
    )

    print("\nMacro-F1 global de la cascada:", f1_cascada)

    # Liberar memoria GPU antes de evaluar otra combinación
    del modelo_n1
    del modelo_n2
    del tokenizer_n1
    del tokenizer_n2

    gc.collect()
    torch.cuda.empty_cache()

    return f1_cascada, predicciones

In [ ]:
# =====================================================
# 29. PREPARAR TEST_DU PARA LAS CASCADAS
# =====================================================

textos_test_DU = test_DU["Text"].astype(str).tolist()
etiquetas_test_DU = test_DU["Label"].astype(int).tolist()

print("Textos de Test DU:", len(textos_test_DU))
print("Etiquetas de Test DU:", len(etiquetas_test_DU))

In [ ]:
# =====================================================
# 30. COMPROBAR QUE EXISTEN TODOS LOS MODELOS
# =====================================================

modelos_necesarios = [
    "RoBERTa_DATD",
    "RoBERTa_SDCNL",
    "RoBERTa_DU",
    "DeBERTa_DATD",
    "DeBERTa_SDCNL",
    "DeBERTa_DU"
]

faltan = [
    modelo
    for modelo in modelos_necesarios
    if not os.path.isdir(rutas_modelos[modelo])
]

if faltan:
    raise FileNotFoundError(
        "Faltan modelos definitivos antes de ejecutar las cascadas: "
        + ", ".join(faltan)
    )

print("Los seis modelos definitivos están disponibles.")

In [ ]:
# =====================================================
# 31. EVALUACIÓN DE LAS CUATRO CASCADAS
# =====================================================

# -----------------------------------------------------
# CASCADA 1
# RoBERTa → RoBERTa
# -----------------------------------------------------

f1_cascada_RR, pred_cascada_RR = evaluar_cascada(
    "RoBERTa",
    rutas_modelos["RoBERTa_DATD"],
    "RoBERTa",
    rutas_modelos["RoBERTa_SDCNL"],
    textos_test_DU,
    etiquetas_test_DU
)


# -----------------------------------------------------
# CASCADA 2
# RoBERTa → DeBERTa
# -----------------------------------------------------

f1_cascada_RD, pred_cascada_RD = evaluar_cascada(
    "RoBERTa",
    rutas_modelos["RoBERTa_DATD"],
    "DeBERTa",
    rutas_modelos["DeBERTa_SDCNL"],
    textos_test_DU,
    etiquetas_test_DU
)


# -----------------------------------------------------
# CASCADA 3
# DeBERTa → RoBERTa
# -----------------------------------------------------

f1_cascada_DR, pred_cascada_DR = evaluar_cascada(
    "DeBERTa",
    rutas_modelos["DeBERTa_DATD"],
    "RoBERTa",
    rutas_modelos["RoBERTa_SDCNL"],
    textos_test_DU,
    etiquetas_test_DU
)


# -----------------------------------------------------
# CASCADA 4
# DeBERTa → DeBERTa
# -----------------------------------------------------

f1_cascada_DD, pred_cascada_DD = evaluar_cascada(
    "DeBERTa",
    rutas_modelos["DeBERTa_DATD"],
    "DeBERTa",
    rutas_modelos["DeBERTa_SDCNL"],
    textos_test_DU,
    etiquetas_test_DU
)

print("\nLas cuatro cascadas han sido evaluadas.")

In [ ]:
# =====================================================
# 32. TABLA FINAL: MODELO DIRECTO VS CASCADAS
# =====================================================

# Cargar los resultados individuales ya guardados
resultados_roberta = pd.read_csv(
    RUTA_RESULTADOS + "resultados_modelos_individuales_RoBERTa.csv"
)

resultados_deberta = pd.read_csv(
    RUTA_RESULTADOS + "resultados_modelos_individuales_DeBERTa.csv"
)

# Mostrar las columnas disponibles para comprobarlas
print("Columnas RoBERTa:", resultados_roberta.columns.tolist())
print("Columnas DeBERTa:", resultados_deberta.columns.tolist())

# Obtener el F1 de Test de los modelos directos entrenados con DU
f1_directo_roberta = resultados_roberta.loc[
    resultados_roberta["Dataset_Test"] == "test_DU",
    "F1_Test"
].iloc[0]

f1_directo_deberta = resultados_deberta.loc[
    resultados_deberta["Dataset_Test"] == "test_DU",
    "F1_Test"
].iloc[0]

# Crear la tabla comparativa final
tabla_final = pd.DataFrame({
    "Arquitectura": [
        "Directo",
        "Directo",
        "Cascada",
        "Cascada",
        "Cascada",
        "Cascada"
    ],

    "Nivel_1": [
        "RoBERTa",
        "DeBERTa",
        "RoBERTa",
        "RoBERTa",
        "DeBERTa",
        "DeBERTa"
    ],

    "Nivel_2": [
        "-",
        "-",
        "RoBERTa",
        "DeBERTa",
        "RoBERTa",
        "DeBERTa"
    ],

    "F1_Test_DU": [
        f1_directo_roberta,
        f1_directo_deberta,
        f1_cascada_RR,
        f1_cascada_RD,
        f1_cascada_DR,
        f1_cascada_DD
    ]
})

print("\n========================================")
print("COMPARACIÓN FINAL")
print("========================================")

display(tabla_final)

In [ ]:
# =====================================================
# 33. GUARDAR TABLA COMPARATIVA FINAL
# =====================================================

tabla_final.to_csv(
    RUTA_RESULTADOS + "comparacion_final_directo_vs_cascada.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tabla comparativa final guardada correctamente.")

# BLOQUE 7. DIAGNÓSTICOS DE LOS RESULTADOS

En este bloque se analiza el comportamiento de los modelos entrenados con el objetivo de explicar las diferencias observadas entre el enfoque directo y el sistema en cascada.

No se realiza ningún nuevo entrenamiento. Los análisis utilizan exclusivamente los modelos definitivos ya guardados y el conjunto `test_DU`.

In [ ]:
# =====================================================
# DIAGNÓSTICO 1
# RENDIMIENTO DEL NIVEL 1 SOBRE TEST DU
# =====================================================
# ¿El Nivel 1 está identificando correctamente quién está sano y quién está enfermo cuando recibe los textos reales de test_DU?

# Para poder evaluar correctamente al Nivel 1 sobre test_DU, necesitamos convertir las etiquetas finales de DU al formato binario que conoce 
# el modelo de Nivel 1
#
# DU:
# 0 = sano      -> 0
# 1 = depresión -> 1
# 2 = suicidio  -> 1
#
# Nivel 1:
# 0 = sano
# 1 = enfermo

y_real_n1 = [
    0 if etiqueta == 0 else 1
    for etiqueta in etiquetas_test_DU
]


def diagnosticar_nivel1(
    nombre_modelo_n1,
    ruta_modelo_n1
):

    # Cargar el modelo entrenado de Nivel 1
    modelo_n1, tokenizer_n1 = cargar_modelo_entrenado(
        ruta_modelo_n1
    )

    # Predecir todo test_DU
    pred_n1 = predecir_textos(
        modelo_n1,
        tokenizer_n1,
        textos_test_DU
    )

    # F1 específico de la clase Enfermo
    f1_enfermo = f1_score(
        y_real_n1,
        pred_n1,
        average="binary",
        pos_label=1
    )

    # Macro-F1 de las dos clases
    f1_macro = f1_score(
        y_real_n1,
        pred_n1,
        average="macro"
    )

    print("\n========================================")
    print("NIVEL 1 SOBRE TEST DU:", nombre_modelo_n1)
    print("========================================")

    print("F1 clase Enfermo:", f1_enfermo)
    print("Macro-F1:", f1_macro)

    print("\nMatriz de confusión:")
    print(
        confusion_matrix(
            y_real_n1,
            pred_n1
        )
    )

    print("\nInforme de clasificación:")

    print(
        classification_report(
            y_real_n1,
            pred_n1,
            target_names=[
                "Sano",
                "Enfermo"
            ],
            digits=4
        )
    )

    # Liberar memoria
    del modelo_n1
    del tokenizer_n1

    gc.collect()
    torch.cuda.empty_cache()

    # Devolver también las predicciones para reutilizarlas
    return pred_n1, f1_enfermo, f1_macro


# -----------------------------------------------------
# RoBERTa
# -----------------------------------------------------

pred_n1_roberta, f1_enfermo_n1_roberta, f1_macro_n1_roberta = (
    diagnosticar_nivel1(
        "RoBERTa",
        rutas_modelos["RoBERTa_DATD"]
    )
)


# -----------------------------------------------------
# DeBERTa
# -----------------------------------------------------

pred_n1_deberta, f1_enfermo_n1_deberta, f1_macro_n1_deberta = (
    diagnosticar_nivel1(
        "DeBERTa",
        rutas_modelos["DeBERTa_DATD"]
    )
)

In [ ]:
# =====================================================
# RESUMEN DEL DIAGNÓSTICO DEL NIVEL 1
# =====================================================

resumen_nivel1 = pd.DataFrame({
    "Modelo": [
        "RoBERTa",
        "DeBERTa"
    ],

    "F1_Enfermo": [
        f1_enfermo_n1_roberta,
        f1_enfermo_n1_deberta
    ],

    "Macro_F1": [
        f1_macro_n1_roberta,
        f1_macro_n1_deberta
    ]
})

print("\n========================================")
print("RESUMEN DEL NIVEL 1")
print("========================================")

display(resumen_nivel1)

# Guardar resultados
resumen_nivel1.to_csv(
    os.path.join(
        RUTA_RESULTADOS,
        "diagnostico_nivel1_test_DU.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("Resumen del Nivel 1 guardado correctamente.")

In [ ]:
# =====================================================
# DIAGNÓSTICO 2
# RENDIMIENTO DEL NIVEL 2 SOBRE TEST DU
# =====================================================
# Si los textos enfermos consiguieran llegar correctamente al Nivel 2, ¿el clasificador depresión/suicidio funciona bien?

# Seleccionar únicamente los casos que realmente
# son Depresión o Suicidio

indices_enfermos_reales = [
    i
    for i, etiqueta in enumerate(etiquetas_test_DU)
    if etiqueta in [1, 2]
]

textos_enfermos_reales = [
    textos_test_DU[i]
    for i in indices_enfermos_reales
]


# Recodificar:
#
# DU:
# 1 = depresión
# 2 = suicidio
#
# Nivel 2:
# 0 = depresión
# 1 = suicidio

y_real_n2 = [
    etiquetas_test_DU[i] - 1
    for i in indices_enfermos_reales
]


def diagnosticar_nivel2(
    nombre_modelo_n2,
    ruta_modelo_n2
):

    # Cargar modelo entrenado de Nivel 2
    modelo_n2, tokenizer_n2 = cargar_modelo_entrenado(
        ruta_modelo_n2
    )

    # Predecir únicamente los enfermos reales
    pred_n2 = predecir_textos(
        modelo_n2,
        tokenizer_n2,
        textos_enfermos_reales
    )

    # F1 específico de la clase Suicidio
    f1_suicidio = f1_score(
        y_real_n2,
        pred_n2,
        average="binary",
        pos_label=1
    )

    # Macro-F1 de Depresión y Suicidio
    f1_macro = f1_score(
        y_real_n2,
        pred_n2,
        average="macro"
    )

    print("\n========================================")
    print("NIVEL 2 SOBRE TEST DU:", nombre_modelo_n2)
    print("========================================")

    print("F1 clase Suicidio:", f1_suicidio)
    print("Macro-F1:", f1_macro)

    print("\nMatriz de confusión:")
    print(
        confusion_matrix(
            y_real_n2,
            pred_n2
        )
    )

    print("\nInforme de clasificación:")

    print(
        classification_report(
            y_real_n2,
            pred_n2,
            target_names=[
                "Depresión",
                "Suicidio"
            ],
            digits=4
        )
    )

    # Liberar memoria
    del modelo_n2
    del tokenizer_n2

    gc.collect()
    torch.cuda.empty_cache()

    return pred_n2, f1_suicidio, f1_macro


# -----------------------------------------------------
# RoBERTa
# -----------------------------------------------------

pred_n2_roberta, f1_suicidio_roberta, f1_macro_n2_roberta = (
    diagnosticar_nivel2(
        "RoBERTa",
        rutas_modelos["RoBERTa_SDCNL"]
    )
)


# -----------------------------------------------------
# DeBERTa
# -----------------------------------------------------

pred_n2_deberta, f1_suicidio_deberta, f1_macro_n2_deberta = (
    diagnosticar_nivel2(
        "DeBERTa",
        rutas_modelos["DeBERTa_SDCNL"]
    )
)

In [ ]:
# =====================================================
# RESUMEN DEL DIAGNÓSTICO DEL NIVEL 2
# =====================================================

resumen_nivel2 = pd.DataFrame({
    "Modelo": [
        "RoBERTa",
        "DeBERTa"
    ],

    "F1_Suicidio": [
        f1_suicidio_roberta,
        f1_suicidio_deberta
    ],

    "Macro_F1": [
        f1_macro_n2_roberta,
        f1_macro_n2_deberta
    ]
})

print("\n========================================")
print("RESUMEN DEL NIVEL 2")
print("========================================")

display(resumen_nivel2)

resumen_nivel2.to_csv(
    os.path.join(
        RUTA_RESULTADOS,
        "diagnostico_nivel2_test_DU.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("Resumen del Nivel 2 guardado correctamente.")

In [ ]:
# =====================================================
# DIAGNÓSTICO 3
# RENDIMIENTO DEL NIVEL 1 SEGÚN LA PROCEDENCIA
# =====================================================
# ¿El Nivel 1 se equivoca igual con todos los textos de test_DU, o se equivoca sobre todo con textos procedentes de alguna base concreta?


def analizar_nivel1_por_source(
    nombre_modelo,
    predicciones_n1
):

    # Crear DataFrame con:
    # - procedencia
    # - etiqueta real
    # - predicción

    diagnostico = pd.DataFrame({
        "Source": test_DU["Source"].values,
        "Real": y_real_n1,
        "Prediccion": predicciones_n1
    })

    resultados = []

    # Analizar cada dataset de procedencia
    for source in diagnostico["Source"].unique():

        datos = diagnostico[
            diagnostico["Source"] == source
        ]

        # Verdaderos negativos
        tn = (
            (datos["Real"] == 0) &
            (datos["Prediccion"] == 0)
        ).sum()

        # Falsos positivos
        fp = (
            (datos["Real"] == 0) &
            (datos["Prediccion"] == 1)
        ).sum()

        # Falsos negativos
        fn = (
            (datos["Real"] == 1) &
            (datos["Prediccion"] == 0)
        ).sum()

        # Verdaderos positivos
        tp = (
            (datos["Real"] == 1) &
            (datos["Prediccion"] == 1)
        ).sum()

        enfermos_reales = (
            datos["Real"] == 1
        ).sum()

        enfermos_detectados = tp

        # Si existen enfermos reales en esa fuente,
        # calcular F1 de la clase Enfermo.
        #
        # DepressionX puede no contener positivos,
        # por lo que en ese caso F1 no es aplicable.

        if enfermos_reales > 0:

            f1_enfermo = f1_score(
                datos["Real"],
                datos["Prediccion"],
                average="binary",
                pos_label=1,
                zero_division=0
            )

        else:

            f1_enfermo = np.nan

        resultados.append({
            "Modelo": nombre_modelo,
            "Source": source,
            "Total": len(datos),
            "Enfermos_reales": enfermos_reales,
            "Enfermos_detectados": enfermos_detectados,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "F1_Enfermo": f1_enfermo
        })

    return pd.DataFrame(resultados)


# -----------------------------------------------------
# RoBERTa
# -----------------------------------------------------

tabla_source_roberta = analizar_nivel1_por_source(
    "RoBERTa",
    pred_n1_roberta
)


# -----------------------------------------------------
# DeBERTa
# -----------------------------------------------------

tabla_source_deberta = analizar_nivel1_por_source(
    "DeBERTa",
    pred_n1_deberta
)


# Unir ambas tablas
tabla_source_completa = pd.concat(
    [
        tabla_source_roberta,
        tabla_source_deberta
    ],
    ignore_index=True
)


print("\n========================================")
print("RENDIMIENTO DEL NIVEL 1 POR SOURCE")
print("========================================")

display(tabla_source_completa)

In [ ]:
# =====================================================
# GUARDAR DIAGNÓSTICO DEL NIVEL 1 POR SOURCE
# =====================================================

tabla_source_completa.to_csv(
    os.path.join(
        RUTA_RESULTADOS,
        "diagnostico_nivel1_por_source.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print(
    "Diagnóstico del Nivel 1 por Source "
    "guardado correctamente."
)

In [ ]:
# =====================================================
# FIGURA 1
# FALSOS NEGATIVOS DEL NIVEL 1 POR SOURCE
# =====================================================

# Nos interesan especialmente los falsos negativos:
# enfermos reales que Nivel 1 clasifica como sanos.

tabla_fn = tabla_source_completa.pivot(
    index="Source",
    columns="Modelo",
    values="FN"
)

ax = tabla_fn.plot(
    kind="bar",
    figsize=(9, 6)
)

ax.set_title(
    "Falsos negativos del Nivel 1 según la procedencia"
)

ax.set_xlabel(
    "Dataset de procedencia"
)

ax.set_ylabel(
    "Número de falsos negativos"
)

plt.xticks(
    rotation=0
)

plt.legend(
    title="Transformer"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        RUTA_RESULTADOS,
        "falsos_negativos_nivel1_por_source.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =====================================================
# DIAGNÓSTICO 4
# COMPARACIÓN DETALLADA:
# MEJOR MODELO DIRECTO VS MEJOR CASCADA
# =====================================================
# ¿En qué clases concretas se diferencia el rendimiento del mejor modelo directo respecto a la mejor cascada?
#
# Mejor modelo directo:
#   RoBERTa-DU
#
# Mejor cascada:
#   DeBERTa-DATD -> RoBERTa-SDCNL
#
# Ambos sistemas se comparan sobre el mismo test_DU
# y utilizando las mismas clases finales:
#
#   0 = sano
#   1 = depresión
#   2 = suicidio
#
# Para realizar la comparación necesitamos las predicciones de ambos sistemas.
# Las predicciones de la mejor cascada ya fueron obtenidas anteriormente y están guardadas en: pred_cascada_DR
# Por tanto, primero obtenemos las predicciones del mejor modelo directo y después calculamos las métricas por clase.


# =====================================================
# 4.1 PREDICCIONES DEL MEJOR MODELO DIRECTO
# ROBERTA-DU
# =====================================================

modelo_directo, tokenizer_directo = cargar_modelo_entrenado(
    rutas_modelos["RoBERTa_DU"]
)

pred_directo_roberta = predecir_textos(
    modelo_directo,
    tokenizer_directo,
    textos_test_DU
)

# Liberar memoria
del modelo_directo
del tokenizer_directo

gc.collect()
torch.cuda.empty_cache()

print(
    "Predicciones del mejor modelo directo "
    "RoBERTa-DU obtenidas correctamente."
)


# =====================================================
# 4.2 MÉTRICAS POR CLASE
# MEJOR DIRECTO VS MEJOR CASCADA
# =====================================================

nombres_clases = [
    "Sano",
    "Depresión",
    "Suicidio"
]


# -----------------------------------------------------
# MEJOR MODELO DIRECTO
# RoBERTa-DU
# -----------------------------------------------------

reporte_directo = classification_report(
    etiquetas_test_DU,
    pred_directo_roberta,
    labels=[0, 1, 2],
    target_names=nombres_clases,
    output_dict=True,
    digits=4
)

tabla_directo = pd.DataFrame(
    reporte_directo
).transpose()


# -----------------------------------------------------
# MEJOR CASCADA
# DeBERTa-DATD -> RoBERTa-SDCNL
# -----------------------------------------------------

reporte_cascada = classification_report(
    etiquetas_test_DU,
    pred_cascada_DR,
    labels=[0, 1, 2],
    target_names=nombres_clases,
    output_dict=True,
    digits=4
)

tabla_cascada = pd.DataFrame(
    reporte_cascada
).transpose()


# =====================================================
# MOSTRAR RESULTADOS
# =====================================================

print("\n========================================")
print("MEJOR MODELO DIRECTO")
print("RoBERTa-DU")
print("========================================")

display(tabla_directo)


print("\n========================================")
print("MEJOR CASCADA")
print("DeBERTa-DATD -> RoBERTa-SDCNL")
print("========================================")

display(tabla_cascada)

In [ ]:
# =====================================================
# 4.3 GUARDAR MÉTRICAS POR CLASE
# =====================================================

tabla_directo.to_csv(
    os.path.join(
        RUTA_RESULTADOS,
        "metricas_por_clase_RoBERTa_Directo.csv"
    ),
    encoding="utf-8-sig"
)

tabla_cascada.to_csv(
    os.path.join(
        RUTA_RESULTADOS,
        "metricas_por_clase_Cascada_DeBERTa_RoBERTa.csv"
    ),
    encoding="utf-8-sig"
)

print(
    "Métricas por clase guardadas correctamente."
)

In [ ]:
# =========================================================================================
# 4.4 TABLA COMPARATIVA DE MÉTRICAS POR CLASE - MEJOR MODELO DIRECTO VS MEJOR CASCADA
# =========================================================================================

filas = []

for clase in nombres_clases:

    # -------------------------------------------------
    # Modelo directo
    # -------------------------------------------------

    filas.append({
        "Arquitectura": "RoBERTa directo",
        "Clase": clase,
        "Precision": tabla_directo.loc[
            clase,
            "precision"
        ],
        "Recall": tabla_directo.loc[
            clase,
            "recall"
        ],
        "F1": tabla_directo.loc[
            clase,
            "f1-score"
        ]
    })

    # -------------------------------------------------
    # Mejor cascada
    # -------------------------------------------------

    filas.append({
        "Arquitectura": "DeBERTa → RoBERTa",
        "Clase": clase,
        "Precision": tabla_cascada.loc[
            clase,
            "precision"
        ],
        "Recall": tabla_cascada.loc[
            clase,
            "recall"
        ],
        "F1": tabla_cascada.loc[
            clase,
            "f1-score"
        ]
    })


tabla_comparacion_clases = pd.DataFrame(
    filas
)


print("\n========================================")
print("COMPARACIÓN DE MÉTRICAS POR CLASE")
print("========================================")

display(tabla_comparacion_clases)


# Guardar tabla
tabla_comparacion_clases.to_csv(
    os.path.join(
        RUTA_RESULTADOS,
        "comparacion_metricas_por_clase.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print(
    "Tabla comparativa por clase "
    "guardada correctamente."
)

In [ ]:
# =====================================================
# 4.5 MATRIZ DE CONFUSIÓN DEL MEJOR MODELO DIRECTO
# ROBERTA-DU
# =====================================================
# OBJETIVO: Mostrar cómo clasifica el mejor modelo directo cada una de las tres clases finales de test_DU:
#
# Esto permite identificar:
# - cuántos textos sanos se clasifican como depresión o suicidio
# - cuántos textos depresivos se confunden con sano o suicidio
# - cuántos textos suicidas se clasifican como sano o depresión.
#
# Esta figura se utilizará posteriormente para comparar visualmente los errores del mejor modelo directo con los errores de la mejor cascada.
# =====================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)

ConfusionMatrixDisplay.from_predictions(
    etiquetas_test_DU,
    pred_directo_roberta,
    labels=[0, 1, 2],
    display_labels=nombres_clases,
    values_format="d",
    ax=ax
)

ax.set_title(
    "Matriz de confusión - RoBERTa directo"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        RUTA_RESULTADOS,
        "matriz_confusion_RoBERTa_Directo.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =====================================================
# 4.6 MATRIZ DE CONFUSIÓN DE LA MEJOR CASCADA
# DEBERTA-DATD → ROBERTA-SDCNL
# =====================================================
# OBJETIVO: Mostrar cómo clasifica la mejor arquitectura en cascada cada una de las tres clases finales de test_DU:
# La cascada utilizada es:
#   Nivel 1: DeBERTa-DATD
#            Sano vs Enfermo
#   Nivel 2: RoBERTa-SDCNL
#            Depresión vs Suicidio

# Esta figura es especialmente importante para analizar
# la propagación de errores de la arquitectura en cascada.
#
# Si un caso de Depresión o Suicidio es clasificado como
# Sano por el Nivel 1, el texto no llega al Nivel 2 y,
# por tanto, ese error ya no puede ser corregido.
#
# La comparación de esta matriz con la matriz del
# modelo directo RoBERTa-DU permite observar qué tipos
# de errores aumentan o disminuyen al utilizar la cascada.
# =====================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)

ConfusionMatrixDisplay.from_predictions(
    etiquetas_test_DU,
    pred_cascada_DR,
    labels=[0, 1, 2],
    display_labels=nombres_clases,
    values_format="d",
    ax=ax
)

ax.set_title(
    "Matriz de confusión - Cascada DeBERTa → RoBERTa"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        RUTA_RESULTADOS,
        "matriz_confusion_Cascada_DeBERTa_RoBERTa.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =====================================================
# 4.7 COMPARACIÓN GRÁFICA DEL F1 POR CLASE
# MEJOR MODELO DIRECTO VS MEJOR CASCADA
# =====================================================
# OBJETIVO: Comparar visualmente el F1-score obtenido por el mejor modelo directo y la mejor cascada para cada una de las tres clases finales:

# La figura utiliza las métricas calculadas anteriormente
# y permite identificar de forma visual en qué clases
# existe una mayor diferencia de rendimiento entre:
#
#   - Mejor modelo directo:
#     RoBERTa-DU
#
#   - Mejor cascada:
#     DeBERTa-DATD -> RoBERTa-SDCNL
# =====================================================

tabla_f1 = tabla_comparacion_clases.pivot(
    index="Clase",
    columns="Arquitectura",
    values="F1"
)

ax = tabla_f1.plot(
    kind="bar",
    figsize=(9, 6)
)

ax.set_title(
    "F1-score por clase: "
    "modelo directo frente a cascada"
)

ax.set_xlabel(
    "Clase"
)

ax.set_ylabel(
    "F1-score"
)

ax.set_ylim(
    0,
    1
)

plt.xticks(
    rotation=0
)

plt.legend(
    title="Arquitectura"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        RUTA_RESULTADOS,
        "comparacion_F1_por_clase.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =====================================================
# RESUMEN FINAL DEL BLOQUE DE DIAGNÓSTICO
# =====================================================
#
# Esta tabla recopila los principales resultados obtenidos
# durante el análisis diagnóstico.
#
# IMPORTANTE:
# Los valores de Macro-F1 de Nivel 1, Nivel 2 y del
# problema final NO deben compararse directamente entre sí,
# ya que corresponden a tareas de clasificación diferentes.
#
# - Nivel 1: Sano vs Enfermo
# - Nivel 2: Depresión vs Suicidio
# - Resultado final: Sano vs Depresión vs Suicidio
#
# La tabla se utiliza únicamente como resumen de los
# diferentes análisis realizados.

resumen_diagnostico = pd.DataFrame({

    "Análisis": [
        "Nivel 1 - RoBERTa",
        "Nivel 1 - DeBERTa",
        "Nivel 2 - RoBERTa",
        "Nivel 2 - DeBERTa",
        "Modelo directo - RoBERTa",
        "Mejor cascada - DeBERTa → RoBERTa"
    ],

    "Tarea evaluada": [
        "Sano vs Enfermo",
        "Sano vs Enfermo",
        "Depresión vs Suicidio",
        "Depresión vs Suicidio",
        "Sano vs Depresión vs Suicidio",
        "Sano vs Depresión vs Suicidio"
    ],

    "Métrica": [
        "Macro-F1",
        "Macro-F1",
        "Macro-F1",
        "Macro-F1",
        "Macro-F1",
        "Macro-F1"
    ],

    "Valor": [
        f1_macro_n1_roberta,
        f1_macro_n1_deberta,
        f1_macro_n2_roberta,
        f1_macro_n2_deberta,
        tabla_directo.loc["macro avg", "f1-score"],
        tabla_cascada.loc["macro avg", "f1-score"]
    ]
})


print("\n========================================")
print("RESUMEN FINAL DEL BLOQUE DE DIAGNÓSTICO")
print("========================================")

display(resumen_diagnostico)


resumen_diagnostico.to_csv(
    os.path.join(
        RUTA_RESULTADOS,
        "resumen_final_diagnostico.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print(
    "Resumen final del bloque de diagnóstico "
    "guardado correctamente."
)